# Advance Crime Data Pipeline

## Stage 4: Aggregation Layer — Gold

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 4 of 5: Aggregation <br>
**Medallion Layer:** Gold 🥇 <br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Cumbria <br>
**Authors:** Group 1 <br>
**Last Updated:** 21 May 2026

This notebook aggregates the enriched Silver dataset to a single, consistent reporting grain for BI consumption. All aggregation decisions and metric calculations are documented below. The output is a clean, BI-ready dataset ready for Power BI.

---

### Purpose

This notebook forms the **Gold layer** of the medallion pipeline. It aggregates enriched crime records to the reporting grain and calculates normalised metrics, enabling analysts to answer the following questions in Power BI:

1. How is overall crime changing over time for each police force?
2. What crime types drive crime volume and how does the mix change over time?
3. Are there seasonal patterns or anomalous spikes at force or category level?
4. How do selected police forces compare head-to-head on key metrics?
5. How do forces compare on normalised crime rates (per 1,000 residents)?
6. How does crime data relate to population and socioeconomic context?
7. Do any areas show repeat seasonal patterns?

---

### How to Use

**Running the notebook end to end.** Run all cells in order from top to bottom. The notebook is designed to be fully repeatable. No manual steps are required between cells.

**Power BI:** Connect directly to `CRIME_PIPELINE.REPORTING.GOLD_CRIME_REPORTING` using the Snowflake connector in Power BI Desktop.

## 1. Environment Setup

This section sets up the Snowflake and Python environment required for the pipeline. It creates the necessary **warehouse**, **database**, and **schemas** if they do not already exist.

In [ ]:
%%sql -r dataframe_1
-- Use the SYSADMIN role and CRIME_WH warehouse
USE ROLE SYSADMIN;
USE WAREHOUSE CRIME_WH;

-- Create the database
CREATE DATABASE IF NOT EXISTS CRIME_PIPELINE;

-- Create schemas for each pipeline stage
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.RAW;        -- Bronze
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.CLEAN;      -- Silver
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.REPORTING;  -- Gold

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

session = get_active_session()
session.sql("USE DATABASE CRIME_PIPELINE").collect()
session.sql("USE SCHEMA REPORTING").collect()

# Source and destination table references
ENRICHED_TABLE = "CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED"
GOLD_TABLE     = "GOLD_CRIME_REPORTING"

# Confirm session is pointing at the correct database and schema
print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. Read from Enriched Silver Table and Initial Inspection

Load the fully enriched dataset from `SILVER_CRIME_ENRICHED`. This is the hand-off from the feature engineering layer — no further cleaning is applied here.

In [ ]:
# Read enriched Silver table
crime = session.table(ENRICHED_TABLE).to_pandas()

# Standardise column names: strip whitespace, lowercase, replace spaces with underscores
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]

# Record baseline count -- used in reconciliation report
baseline_count = len(crime)

print("Enriched Silver rows :", len(crime))
print("Columns              :", crime.columns.tolist())

In [ ]:
# Visual inspection of the first five rows to confirm structure
crime.head()

## 3. Pre-Aggregation Preparation

In this step, pipeline provenance columns that carry no analytical value are removed before aggregation. Exposing internal tracking fields to BI consumers would add noise to the reporting dataset.

Five columns are dropped: <br>
1. **source_file** <br>
2. **source_month** <br>
3. **lsoa_name** <br>
4. **falls_within** <br>
5. **crime_id** <br>

**source_file** and **source_month** are pipeline provenance columns used for traceability — not required for reporting. <br>

**lsoa_name** is redundant as `lsoa_code` is sufficient for joining enrichment data. <br>

**falls_within** has been replaced by the cleaner `force_name` column derived in the feature engineering layer. <br>

**crime_id** is an individual record identifier that has no meaning after aggregation.

In [ ]:
cols_to_drop = [
    "source_file",
    "source_month",
    "lsoa_name",
    "falls_within",
    "crime_id"
]

# errors='ignore' ensures the cell is idempotent -- safe to re-run even if columns were already dropped
crime = crime.drop(columns=cols_to_drop, errors="ignore")

print("Columns dropped  :", cols_to_drop)
print("Remaining columns:", crime.columns.tolist())

## 4. Aggregate to Reporting Grain

In this step, individual crime records are aggregated to the reporting grain: `force_name × month × year × month_num × crime_type`.

Each metric is calculated as follows: <br>
- **`crime_count`** — count of all crime records at the grain <br>
- **`population`** — sum of LSOA populations within the force (used for rate calculation) <br>
- **`avg_house_price_2023`** — mean house price across LSOAs at the grain <br>
- **`avg_imd_rank`** — mean IMD rank across LSOAs at the grain

In [ ]:
# Define reporting grain -- the combination of columns that uniquely identifies each row
GRAIN_COLS = ["force_name", "month", "year", "month_num", "crime_type"]

# Aggregate all metrics at reporting grain
gold = crime.groupby(GRAIN_COLS, as_index=False).agg(
    crime_count          = ("lsoa_code",               "count"),
    population           = ("population",               "sum"),
    avg_house_price_2023 = ("average_house_price_2023", "mean"),
    avg_imd_rank         = ("imd_rank",                 "mean"),
    avg_income_rank      = ("income_rank",              "mean"),
    avg_employment_rank  = ("employment_rank",          "mean"),
    avg_health_rank      = ("health_rank",              "mean"),
    avg_crime_rank       = ("crime_rank",               "mean"),
)

# Round float columns for cleaner BI display
float_cols = [c for c in gold.columns if gold[c].dtype == "float64"]
gold[float_cols] = gold[float_cols].round(2)

print(f"Gold rows : {len(gold):,}")
print(f"Columns   : {gold.columns.tolist()}")

## 5. Calculate Normalised Crime Rate

Calculate `crime_rate_per_1000` — the number of crimes per 1,000 residents. This is the key normalised metric that enables fair comparison between forces of different population sizes.

**Formula:** `crime_count / population × 1,000` <br>

Records with null population (suppressed location records) will produce a null rate — this is expected and documented.

In [ ]:
# Calculate crime rate per 1,000 residents
# Null population produces null rate -- expected for suppressed location records
gold["crime_rate_per_1000"] = (
    gold["crime_count"] / gold["population"] * 1000
).round(2)

print("Null crime rates:", gold["crime_rate_per_1000"].isnull().sum())
print(gold[["force_name", "crime_type", "crime_count", "population", "crime_rate_per_1000"]].head(10))

## 6. Gold Validation Report

Confirm the Gold dataset is BI-ready before export. Three checks are performed:
- No duplicate rows at the reporting grain
- No missing values in required reporting fields
- Total crime count matches the Silver baseline — confirms no records were lost during aggregation

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║              GOLD VALIDATION REPORT                         ║")
print("╚══════════════════════════════════════════════════════════════╝")

# Overview stats
print(f"Gold rows              : {len(gold):,}")
print(f"Distinct forces        : {gold['force_name'].nunique()}")
print(f"Distinct months        : {gold['month'].nunique()}")
print(f"Distinct crime types   : {gold['crime_type'].nunique()}")

# 1. Duplicate check at reporting grain
dup_count = gold.duplicated(subset=GRAIN_COLS).sum()
print(f"\nDuplicates at grain    : {dup_count}")
assert dup_count == 0, "Duplicate rows found at reporting grain — investigate before export"

# 2. Crime count reconciliation -- Gold sum should equal Silver baseline row count
total_crimes = gold["crime_count"].sum()
print(f"\nTotal crimes (Gold)    : {total_crimes:,}")
print(f"Baseline rows (Silver) : {baseline_count:,}")
assert total_crimes == baseline_count, \
    f"Crime count mismatch: {total_crimes} vs {baseline_count}"
print("Crime count reconciliation: ✓ PASSED")

# 3. Null check on required reporting fields
required_cols = ["force_name", "month", "year", "month_num", "crime_type", "crime_count"]
null_check = gold[required_cols].isnull().sum()
print(f"\nNull check on required fields:")
print(null_check)
assert null_check.sum() == 0, "Null values found in required reporting fields"

## 7. Inspect Gold Dataset

Final visual inspection and high level statistics before export.

In [ ]:
print("Shape  :", gold.shape)
print("Columns:", gold.columns.tolist())

gold.head(10)

In [ ]:
# High level statistics -- sense check before export
print("── Crime count by force ──")
print(gold.groupby("force_name")["crime_count"].sum().sort_values(ascending=False))

print("\n── Crime count by crime type ──")
print(gold.groupby("crime_type")["crime_count"].sum().sort_values(ascending=False))

print("\n── Crime count by month ──")
print(gold.groupby("month")["crime_count"].sum().sort_index())

## 8. Export to Gold Table

Persist the aggregated reporting dataset to `CRIME_PIPELINE.REPORTING.GOLD_CRIME_REPORTING`. This is the final output of the pipeline — the table Power BI connects to directly.

In [ ]:
# Reset index before writing to suppress non-standard index warning
gold = gold.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=gold,
    table_name=GOLD_TABLE,
    database="CRIME_PIPELINE",
    schema="REPORTING",
    auto_create_table=True,  # creates table if it does not exist
    overwrite=True           # replaces existing data on each run
)

# Verify written row count matches Gold row count
written_count = session.table(f"CRIME_PIPELINE.REPORTING.{GOLD_TABLE}").count()
assert written_count == len(gold), \
    f"Row count mismatch: {written_count} written vs {len(gold)} expected"

print(f"Gold table written successfully")
print(f"Table  : CRIME_PIPELINE.REPORTING.{GOLD_TABLE}")
print(f"Rows   : {written_count:,}")

## 9. Export to CSV

Export the Gold reporting dataset as a CSV for local use or direct import into Power BI.

In [ ]:
gold.to_csv("gold_crime_reporting.csv", index=False)

print("Exported : gold_crime_reporting.csv")
print(f"Rows     : {len(gold):,}")
print(f"Columns  : {len(gold.columns)}")
print(f"\nColumn list: {gold.columns.tolist()}")